# 05 — Long-text handling, ensemble, final submission

| Part | Runs on | Time (estimate) | Output |
|---|---|---|---|
| A. Long texts: the trained mDeBERTa scored with *first 512 tokens* vs *first+last* vs *all chunks (mean)* vs *all chunks (max fusion)* | GPU | ~15 min | rows in `results.csv`, test scores for the best rule |
| B. TF-IDF word+char LogReg refit → validation **and** test probabilities | CPU | ~10 min, ~3.5 GB RAM | `val_scores/05_tfidf.csv`, `test_scores/05_tfidf.csv` |
| C. Ensemble: weighted average + per-class offsets tuned by 5-fold CV on validation; scorecard; diversity | CPU | ~1 min | `submissions/05_ensemble.csv`, `submissions/05_hedge.csv`, `outputs/report_tables.md` |

Members of the ensemble = every model that has **both** `outputs/val_scores/<name>.csv` and `outputs/test_scores/<name>.csv`.
Set `RUN_PART_A = False` to skip the GPU part (C then uses mDeBERTa's first+last scores from notebook 04).

In [1]:
import os
import sys
from pathlib import Path

SRC = next(q for p in [Path.cwd(), *Path.cwd().parents] for q in (p, p / "src") if (q / "config.py").exists())
sys.path.insert(0, str(SRC))

In [2]:
import gc
import itertools
import time

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

from config import LABELS, OUTPUT_DIR, SEED
from utils import load_splits, load_test, log_result, macro_f1, save_submission, save_val_scores

RUN_PART_A = True
LT_MODEL_ID = "microsoft/mdeberta-v3-base"          # backbone of the trained checkpoint
LT_CKPT = OUTPUT_DIR / "04_mdeberta_best.pt"        # written by 04 (best epoch)
MAX_LEN, HEAD_TOKENS, EVAL_BATCH = 512, 128, 32

train_df, val_df = load_splits()
test_df = load_test()
y_va = val_df.label.to_numpy()
(OUTPUT_DIR / "test_scores").mkdir(parents=True, exist_ok=True)
print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

train=38,737  val=10,000  test=25,000


## A. Long-text handling for mDeBERTa (no retraining)

72% of training texts (91% of German ones) are longer than 512 tokens. The trained model is re-scored on the fixed
validation split with four rules; *first+last* is what it was trained with.
* **first 512** — first 510 tokens.
* **first+last** — first 128 + last 382 tokens (from notebook 04).
* **chunks mean** — every consecutive 510-token piece is scored; probabilities averaged.
* **chunks max-fusion** — as *chunks mean*, but P(fusion) = the highest P(fusion) of any piece (a rewrite may sit in
  one part of the text), then renormalised.

In [3]:
if RUN_PART_A:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    FP16 = DEVICE == "cuda"
    tok = AutoTokenizer.from_pretrained(LT_MODEL_ID)
    tok.model_max_length = 10**9
    _raw, _full = tok("x", add_special_tokens=False)["input_ids"], tok("x")["input_ids"]
    _i = next(i for i in range(len(_full)) if _full[i:i + len(_raw)] == _raw)
    PREFIX, SUFFIX = _full[:_i], _full[_i + len(_raw):]
    BODY = MAX_LEN - len(PREFIX) - len(SUFFIX)

    model = AutoModelForSequenceClassification.from_pretrained(LT_MODEL_ID, num_labels=len(LABELS),
                                                               ignore_mismatched_sizes=True)
    model.load_state_dict(torch.load(LT_CKPT, map_location="cpu"))
    model = model.float().to(DEVICE).eval()

    def token_ids(texts: pd.Series, chunk: int = 2_000):
        texts = texts.astype(str).str.slice(0, 30_000)
        for i in range(0, len(texts), chunk):
            yield from tok(texts.iloc[i:i + chunk].tolist(), add_special_tokens=False)["input_ids"]

    @torch.no_grad()
    def probs(seqs: list) -> np.ndarray:
        out = np.zeros((len(seqs), len(LABELS)), dtype=np.float32)
        order = np.argsort([len(s) for s in seqs])
        pad = tok.pad_token_id if tok.pad_token_id is not None else 0
        for b in range(0, len(order), EVAL_BATCH):
            idx = order[b:b + EVAL_BATCH]
            L = max(len(seqs[j]) for j in idx)
            ids = torch.full((len(idx), L), pad, dtype=torch.long)
            att = torch.zeros((len(idx), L), dtype=torch.long)
            for r, j in enumerate(idx):
                ids[r, :len(seqs[j])] = torch.as_tensor(seqs[j]); att[r, :len(seqs[j])] = 1
            with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
                logits = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE)).logits.float()
            out[idx] = torch.softmax(logits, 1).cpu().numpy()
        return out

    def score_rules(texts: pd.Series) -> dict:
        """P(human, ai, fusion) per text under the first-512 and the two chunk rules."""
        heads, pieces, owner = [], [], []
        for k, x in enumerate(token_ids(texts)):
            heads.append(PREFIX + x[:BODY] + SUFFIX)
            for s in range(0, max(1, len(x)), BODY):
                pieces.append(PREFIX + x[s:s + BODY] + SUFFIX); owner.append(k)
        owner = np.array(owner)
        p_piece = probs(pieces)
        n = len(heads)
        mean = np.zeros((n, len(LABELS))); np.add.at(mean, owner, p_piece)
        mean /= np.bincount(owner, minlength=n)[:, None]
        fus = LABELS.index("fusion")
        mx = mean.copy(); mx[:, fus] = 0; np.maximum.at(mx[:, fus], owner, p_piece[:, fus])
        mx /= mx.sum(1, keepdims=True)
        print(f"  {n:,} texts -> {len(pieces):,} chunks ({len(pieces) / n:.2f} per text)")
        return {"first 512": probs(heads), "chunks mean": mean, "chunks max-fusion": mx}

    t0 = time.time()
    rules = score_rules(val_df.text)
    rules["first+last (04)"] = pd.read_csv(OUTPUT_DIR / "val_scores" / "04_mdeberta.csv").set_index("id").loc[val_df.id, LABELS].to_numpy()
    rows = []
    for name, p in rules.items():
        pred = np.array(LABELS)[p.argmax(1)]
        f1c = f1_score(y_va, pred, labels=LABELS, average=None)
        rows.append({"rule": name, "macro_f1": macro_f1(y_va, pred), **{f"f1_{c}": v for c, v in zip(LABELS, f1c)},
                     **{f"macro_f1_{l}": macro_f1(y_va[m], pred[m]) for l in sorted(val_df.lang.unique()) if (m := (val_df.lang == l).to_numpy()).any()}})
        if name != "first+last (04)":
            log_result("05_long_text", f"mdeberta | {name}", y_va, pred, notes="inference-time long-text rule")
    long_text = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
    print(long_text.round(4).to_string(index=False), f"\n[{(time.time() - t0) / 60:.0f} min]")

    BEST_RULE = long_text.iloc[0]["rule"]
    if BEST_RULE != "first+last (04)":           # the best rule becomes an ensemble member with its own test scores
        save_val_scores("05_mdeberta_chunks", val_df.id, rules[BEST_RULE], LABELS)
        te = score_rules(test_df.text)[BEST_RULE]
        pd.DataFrame(te, columns=LABELS).assign(id=test_df.id.values).to_csv(OUTPUT_DIR / "test_scores" / "05_mdeberta_chunks.csv", index=False)
        print(f"saved {BEST_RULE!r} scores as member 05_mdeberta_chunks")
    del model; gc.collect()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
classifier.bias     

  10,000 texts -> 22,251 chunks (2.23 per text)
             rule  macro_f1  f1_human  f1_ai  f1_fusion  macro_f1_de-DE  macro_f1_en-EN
  first+last (04)    0.9055    0.8489 0.9928     0.8749          0.9040          0.9058
      chunks mean    0.8700    0.7783 0.9873     0.8444          0.8125          0.8843
chunks max-fusion    0.8304    0.6909 0.9855     0.8148          0.6845          0.8617
        first 512    0.8282    0.7015 0.9881     0.7951          0.6684          0.8593 
[5 min]


## B. TF-IDF word+char LogReg with test probabilities

Same features and model as the best baseline in 01 (`word_cased_punct` + `char_wb_3_5`, LogReg C=4, no class weights:
0.8601 on validation). Notebook 01 saved only its validation scores, so it is refit here to get test scores too.
Skipped if both score files already exist.

In [4]:
if not ((OUTPUT_DIR / "val_scores" / "05_tfidf.csv").exists() and (OUTPUT_DIR / "test_scores" / "05_tfidf.csv").exists()):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from utils import hstack_csr

    MAX_CHARS = 4_000
    TOKEN_PATTERN = r"(?u)\b\w+\b|[^\w\s]"
    VECS = [dict(analyzer="word", ngram_range=(1, 2), min_df=3, max_features=200_000, lowercase=False, token_pattern=TOKEN_PATTERN),
            dict(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_features=300_000, lowercase=False)]
    clip = lambda s: s.astype(str).str.slice(0, MAX_CHARS)
    t0 = time.time()
    blocks = {"tr": [], "va": [], "te": []}
    for kw in VECS:
        vec = TfidfVectorizer(sublinear_tf=True, dtype=np.float32, **kw)
        blocks["tr"].append(vec.fit_transform(clip(train_df.text)))      # fit on training rows only
        vec.stop_words_ = None
        blocks["va"].append(vec.transform(clip(val_df.text)))
        blocks["te"].append(vec.transform(clip(test_df.text)))
        del vec; gc.collect()
    X = {}
    for k in list(blocks):                       # stack one split at a time and free its blocks right away
        X[k] = hstack_csr(blocks.pop(k)); gc.collect()
    lr = LogisticRegression(C=4, max_iter=2000).fit(X["tr"], train_df.label)
    p_va = lr.predict_proba(X["va"])[:, [list(lr.classes_).index(c) for c in LABELS]]
    p_te = lr.predict_proba(X["te"])[:, [list(lr.classes_).index(c) for c in LABELS]]
    save_val_scores("05_tfidf", val_df.id, p_va, LABELS)
    pd.DataFrame(p_te, columns=LABELS).assign(id=test_df.id.values).to_csv(OUTPUT_DIR / "test_scores" / "05_tfidf.csv", index=False)
    print(f"TF-IDF LogReg: val macro-F1 = {macro_f1(y_va, np.array(LABELS)[p_va.argmax(1)]):.4f}  [{(time.time() - t0) / 60:.0f} min]")
    del X, lr; gc.collect()
else:
    print("05_tfidf scores exist - skipped")

TF-IDF LogReg: val macro-F1 = 0.8601  [4 min]


## C. Ensemble and scorecard

**Candidates:** every single member, a weighted average of members, and the weighted average with per-class offsets
(probabilities multiplied by per-class factors, which moves the decision boundaries for macro-F1).
**Honest scores:** weights and offsets are tuned on 4/5 of the validation split and scored on the remaining 1/5,
5 times (folds stratified by domain × label). The reported per-class F1 and confusions use these out-of-fold
predictions. Final weights/offsets are then refit on the whole validation split and applied to test.

In [5]:
names = sorted({p.stem for p in (OUTPUT_DIR / "val_scores").glob("*.csv")} & {p.stem for p in (OUTPUT_DIR / "test_scores").glob("*.csv")})
if "05_mdeberta_chunks" in names and "04_mdeberta" in names:
    names.remove("04_mdeberta")                    # same model, keep only the better long-text rule
V = {n: pd.read_csv(OUTPUT_DIR / "val_scores" / f"{n}.csv").set_index("id").loc[val_df.id, LABELS].to_numpy() for n in names}
T = {n: pd.read_csv(OUTPUT_DIR / "test_scores" / f"{n}.csv").set_index("id").loc[test_df.id, LABELS].to_numpy() for n in names}
print("members:", names)

folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(val_df, val_df.domain + "|" + val_df.label))
f1m = lambda y, p: f1_score(y, np.array(LABELS)[p.argmax(1)], average="macro", labels=LABELS)
W_GRID = [w for w in itertools.product(np.arange(0, 1.01, 0.1), repeat=len(names)) if abs(sum(w) - 1) < 1e-6] if len(names) > 1 else [(1.0,)]
B_GRID = [np.array([bh, 1.0, bf]) for bh in np.arange(0.6, 1.61, 0.1) for bf in np.arange(0.6, 1.61, 0.1)]   # human, ai, fusion


def blend(P, w):
    return sum(wi * P[n] for wi, n in zip(w, names))


def fit(idx, use_bias, members=None):
    """Best weights (and per-class factors) on rows idx."""
    grid = [w for w in W_GRID if members is None or all(wi == 0 for wi, n in zip(w, names) if n not in members)]
    w = max(grid, key=lambda w: f1m(y_va[idx], blend(V, w)[idx]))
    b = max(B_GRID, key=lambda b: f1m(y_va[idx], blend(V, w)[idx] * b)) if use_bias else np.ones(3)
    return w, b


def oof(use_bias, members=None):
    """Out-of-fold probabilities and per-fold macro-F1."""
    P, scores = np.zeros_like(V[names[0]]), []
    for tr, te in folds:
        w, b = fit(tr, use_bias, members)
        P[te] = blend(V, w)[te] * b
        scores.append(f1m(y_va[te], P[te]))
    return P, np.array(scores)


cands = {n: (V[n], np.array([f1m(y_va[te], V[n][te]) for _, te in folds])) for n in names}
if len(names) > 1:
    cands["ensemble: weighted average"] = oof(False)
cands["ensemble: weighted + class offsets" if len(names) > 1 else f"{names[0]} + class offsets"] = oof(True)


def scorecard_row(name, P, s):
    pred = np.array(LABELS)[P.argmax(1)]
    cm = confusion_matrix(y_va, pred, labels=LABELS)
    h, f = LABELS.index("human"), LABELS.index("fusion")
    langs = {f"macro_f1 {l}": macro_f1(y_va[m], pred[m]) for l in sorted(val_df.lang.unique()) if (m := (val_df.lang == l).to_numpy()).any()}
    dom = {d: macro_f1(y_va[m], pred[m]) for d in val_df.domain.unique() if (m := (val_df.domain == d).to_numpy()).any()}
    worst = min(dom, key=dom.get)
    return {"candidate": name, "macro_f1 (5-fold)": f"{s.mean():.4f} ± {s.std():.4f}",
            **{f"f1_{c}": v for c, v in zip(LABELS, f1_score(y_va, pred, labels=LABELS, average=None))},
            "human<->fusion confusion": (cm[h, f] + cm[f, h]) / (cm[h].sum() + cm[f].sum()),
            **langs, "worst domain": f"{worst} {dom[worst]:.3f}"}


scorecard = pd.DataFrame([scorecard_row(n, P, s) for n, (P, s) in cands.items()])
print(scorecard.round(4).to_string(index=False))

members: ['04_mdeberta', '05_tfidf']
                         candidate macro_f1 (5-fold)  f1_human  f1_ai  f1_fusion  human<->fusion confusion  macro_f1 de-DE  macro_f1 en-EN       worst domain
                       04_mdeberta   0.9055 ± 0.0073    0.8489 0.9928     0.8749                    0.1304          0.9040          0.9058 arxiv_papers 0.680
                          05_tfidf   0.8602 ± 0.0077    0.7875 0.9877     0.8052                    0.1919          0.8813          0.8544    gutenberg 0.584
        ensemble: weighted average   0.9164 ± 0.0056    0.8670 0.9938     0.8885                    0.1157          0.9211          0.9151    gutenberg 0.720
ensemble: weighted + class offsets   0.9177 ± 0.0073    0.8697 0.9944     0.8891                    0.1150          0.9265          0.9153 arxiv_papers 0.734


**Diversity.** Error correlation = correlation between two members' 0/1 "wrong" vectors on validation (low = they
fail on different texts, so combining them helps). Leave-one-out = 5-fold macro-F1 of the weighted ensemble minus
the same ensemble without that member (positive = the member earns its place).

In [6]:
wrong = pd.DataFrame({n: (np.array(LABELS)[V[n].argmax(1)] != y_va).astype(float) for n in names})
err_corr = wrong.corr().round(3)
print("Error correlation between members:\n" + err_corr.to_string())
if len(names) > 1:
    full = cands["ensemble: weighted average"][1].mean()
    loo = pd.Series({n: full - oof(False, [m for m in names if m != n])[1].mean() for n in names}, name="leave-one-out gain")
    print("\n" + loo.round(4).to_string())

Error correlation between members:
             04_mdeberta  05_tfidf
04_mdeberta        1.000     0.263
05_tfidf           0.263     1.000

04_mdeberta    0.0563
05_tfidf       0.0109


### Final and hedge submissions
**Final** = the candidate with the best 5-fold macro-F1, refit on the whole validation split and applied to test.
**Hedge** = the best single model as it is (no tuning on validation).

In [7]:
cv_mean = {n: s.mean() for n, (_, s) in cands.items()}
best = max(cv_mean, key=cv_mean.get)
single = max(names, key=lambda n: cv_mean[n])
all_idx = np.arange(len(val_df))
if best in names:
    p_te, w, b = T[best], None, np.ones(3)
else:
    w, b = fit(all_idx, use_bias="offsets" in best)
    p_te = sum(wi * T[n] for wi, n in zip(w, names)) * b
print(f"final  = {best}  (5-fold macro-F1 {cv_mean[best]:.4f})")
if w is not None:
    print("  weights:", {n: round(float(x), 2) for n, x in zip(names, w)}, " class factors:", {c: round(float(x), 2) for c, x in zip(LABELS, b)})
print(f"hedge  = {single}  (5-fold macro-F1 {cv_mean[single]:.4f})")
save_submission(test_df.id, np.array(LABELS)[p_te.argmax(1)], "05_ensemble")
save_submission(test_df.id, np.array(LABELS)[T[single].argmax(1)], "05_hedge")

P_best = cands[best][0]
_ = log_result("05_final", f"FINAL {best}", y_va, np.array(LABELS)[P_best.argmax(1)], notes="out-of-fold predictions on val")

final  = ensemble: weighted + class offsets  (5-fold macro-F1 0.9177)
  weights: {'04_mdeberta': 0.4, '05_tfidf': 0.6}  class factors: {'human': 1.4, 'ai': 1.0, 'fusion': 1.3}
hedge  = 04_mdeberta  (5-fold macro-F1 0.9055)
saved submissions/05_ensemble.csv  rows=25000  label share={'ai': 0.514, 'human': 0.259, 'fusion': 0.227}
saved submissions/05_hedge.csv  rows=25000  label share={'ai': 0.517, 'human': 0.243, 'fusion': 0.24}


### Tables for the report appendix
Written to `outputs/report_tables.md` (markdown; paste into the appendix or convert).

In [8]:
def md(df: pd.DataFrame, index: bool = False) -> str:
    """DataFrame -> markdown table (no extra dependency)."""
    d = df.reset_index() if index else df
    fmt = lambda v: f"{v:.4f}" if isinstance(v, (float, np.floating)) else str(v).replace("|", "/")
    lines = ["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * len(d.columns)]
    return "\n".join(lines + ["| " + " | ".join(fmt(v) for v in row) + " |" for row in d.itertuples(index=False)])


res = pd.read_csv(OUTPUT_DIR / "results.csv")
by_exp = (res.sort_values("macro_f1", ascending=False).groupby("experiment").head(3)
          .sort_values(["experiment", "macro_f1"], ascending=[True, False])[["experiment", "model", "macro_f1", "f1_human", "f1_ai", "f1_fusion"]])
parts = ["## Scorecard (fixed validation split, 5-fold out-of-fold)", md(scorecard),
         "## Error correlation between members", md(err_corr, index=True)]
if len(names) > 1:
    parts += ["## Leave-one-out gain", md(loo.to_frame(), index=True)]
if RUN_PART_A:
    parts += ["## Long-text rules (mDeBERTa, validation)", md(long_text)]
parts += ["## All experiments (top 3 per experiment, validation)", md(by_exp),
          "## Submissions", f"- final: `submissions/05_ensemble.csv` = {best}", f"- hedge: `submissions/05_hedge.csv` = {single}"]
(OUTPUT_DIR / "report_tables.md").write_text("\n\n".join(parts) + "\n")
print("written outputs/report_tables.md")

written outputs/report_tables.md
